In [0]:
dbutils.widgets.text("catalog", "claims_dev")

In [0]:
from functools import reduce
from pyspark.sql import functions as F

catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {catalog}")

tables = ["bronze.claim_events_app", "bronze.claim_events_tpa", "silver.claim_events",
          "silver.fnol_hourly", "gold.fact_claim"]


def table_stats():
    return reduce(lambda a, b: a.unionByName(b), [
        spark.sql(f"DESCRIBE DETAIL {t}").select(
            F.lit(t).alias("table"), "numFiles", "sizeInBytes", "clusteringColumns")
        for t in tables
    ])


display(table_stats())

In [0]:
display(spark.sql("""
    SELECT city, count(*) AS payments, sum(amount) AS paid
    FROM silver.claim_events
    WHERE event_type = 'PAYMENT'
      AND event_ts >= '2026-04-22' AND event_ts < '2026-04-29'
    GROUP BY city
    ORDER BY paid DESC
"""))

In [0]:
display(spark.sql("SELECT count(*) AS rows FROM system.billing.usage"))

In [0]:
display(spark.sql("DESCRIBE HISTORY silver.claim_events").select(
    "version", "timestamp", "operation", "operationParameters", "userName"))

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE silver.claim_events_bench
TBLPROPERTIES ('delta.targetFileSize' = '8mb', 'delta.autoOptimize.autoCompact' = 'false')
AS SELECT e.*, r.id AS copy_id
FROM silver.claim_events e
CROSS JOIN range(400) r
ORDER BY rand()
""")

display(spark.sql("DESCRIBE DETAIL silver.claim_events_bench").select("numFiles", "sizeInBytes", "clusteringColumns"))
display(spark.sql("SELECT count(*) AS rows FROM silver.claim_events_bench"))

In [0]:
display(spark.sql("""
    SELECT city, count(*) AS payments, sum(amount) AS paid
    FROM silver.claim_events_bench
    WHERE event_type = 'PAYMENT'
      AND event_ts >= '2026-04-22' AND event_ts < '2026-04-29'
    GROUP BY city
    ORDER BY paid DESC
"""))

In [0]:
spark.sql("ALTER TABLE silver.claim_events_bench CLUSTER BY (event_ts, city)")
spark.sql("OPTIMIZE silver.claim_events_bench FULL")

display(spark.sql("DESCRIBE DETAIL silver.claim_events_bench").select("numFiles", "sizeInBytes", "clusteringColumns"))

In [0]:
display(spark.sql("""
    SELECT city, count(*) AS payments, sum(amount) AS paid
    FROM silver.claim_events_bench
    WHERE event_type = 'PAYMENT'
      AND event_ts >= '2026-04-22' AND event_ts < '2026-04-29'
    GROUP BY city
    ORDER BY paid DESC
"""))

In [0]:
display(spark.sql("""
    SELECT billing_origin_product, sku_name, round(sum(usage_quantity), 2) AS dbus
    FROM system.billing.usage
    WHERE usage_unit = 'DBU' AND usage_date >= '2026-11-21'
    GROUP BY ALL
    ORDER BY dbus DESC
"""))

In [0]:
display(spark.sql("""
    SELECT usage_date, billing_origin_product, round(sum(usage_quantity), 2) AS dbus
    FROM system.billing.usage
    WHERE usage_unit = 'DBU' AND usage_date >= current_date() - INTERVAL 30 DAYS
    GROUP BY ALL
    ORDER BY usage_date, dbus DESC
"""))